In [0]:

df_silver = spark.table(
    "workspace.greenhouse_silver.production_clean"
)

display(
    df_silver
    .groupBy(
        "production_and_value",
        "estimates",
        "uom"
    )
    .count()
    .orderBy(
        "production_and_value",
        "estimates",
        "uom"
    )
)


production_and_value,estimates,uom,count
Area harvested,Square feet,Square feet,2050
Area harvested,Square metres,Square metres,2050
Average price,Dollars per dozen,Dollars per dozen,584
Average price,Dollars per head,Dollars,110
Average price,Dollars per kilogram,Dollars per kilogram,1775
Average price,Dollars per pound,Dollars per pound,2299
Farm gate value,Dollars,Dollars,3048
Production,Dozens,Dozens,584
Production,Heads,Head,110
Production,Kilograms,Kilograms,1775


In [0]:

from pyspark.sql.functions import col

df_production = df_silver.filter(
    col("production_and_value") == "Production"
)

display(
    df_production
    .groupBy("commodity", "uom")
    .count()
    .orderBy("commodity", "uom")
)


commodity,uom,count
Fresh Chinese vegetables,Kilograms,165
Fresh Chinese vegetables,Pounds,165
Fresh cucumbers [11422152],Dozens,584
Fresh cucumbers [11422152],Kilograms,220
Fresh cucumbers [11422152],Pounds,220
Fresh eggplants (except Chinese eggplants) [114221571],Kilograms,165
Fresh eggplants (except Chinese eggplants) [114221571],Pounds,165
Fresh fine herbs [11422138],Kilograms,165
Fresh fine herbs [11422138],Pounds,165
Fresh green and wax beans [114221621],Kilograms,165


In [0]:

from pyspark.sql.functions import col

# Step 1: Read the saved Silver Delta table
df_silver = spark.table(
    "workspace.greenhouse_silver.production_clean"
)

# Step 2: Select only production measurements
df_production = df_silver.filter(
    col("production_and_value") == "Production"
)

# Step 3: Identify commodities and their units
display(
    df_production
    .groupBy("commodity", "uom")
    .count()
    .orderBy("commodity", "uom")
)


commodity,uom,count
Fresh Chinese vegetables,Kilograms,165
Fresh Chinese vegetables,Pounds,165
Fresh cucumbers [11422152],Dozens,584
Fresh cucumbers [11422152],Kilograms,220
Fresh cucumbers [11422152],Pounds,220
Fresh eggplants (except Chinese eggplants) [114221571],Kilograms,165
Fresh eggplants (except Chinese eggplants) [114221571],Pounds,165
Fresh fine herbs [11422138],Kilograms,165
Fresh fine herbs [11422138],Pounds,165
Fresh green and wax beans [114221621],Kilograms,165


In [0]:

from pyspark.sql.functions import col, countDistinct

df_silver = spark.table(
    "workspace.greenhouse_silver.production_clean"
)

df_mass = df_silver.filter(
    (col("production_and_value") == "Production") &
    (col("uom").isin("Kilograms", "Pounds"))
)

overlapping_units = (
    df_mass
    .groupBy("ref_date", "geo", "commodity")
    .agg(
        countDistinct("uom").alias("unit_count")
    )
    .filter(col("unit_count") > 1)
)

print(
    "Province-year-commodity combinations with both units:",
    overlapping_units.count()
)

display(overlapping_units.limit(10))


Province-year-commodity combinations with both units: 1775


ref_date,geo,commodity,unit_count
2006,Canada,Fresh tomatoes [11422151],2
2006,Canada,Fresh cucumbers [11422152],2
2006,Canada,Fresh peppers [114221541],2
2006,Newfoundland and Labrador,Fresh tomatoes [11422151],2
2006,Newfoundland and Labrador,Fresh cucumbers [11422152],2
2006,Newfoundland and Labrador,Fresh peppers [114221541],2
2006,Prince Edward Island,Fresh tomatoes [11422151],2
2006,Prince Edward Island,Fresh cucumbers [11422152],2
2006,Prince Edward Island,Fresh peppers [114221541],2
2006,Nova Scotia,Fresh tomatoes [11422151],2


In [0]:

from pyspark.sql.functions import (
    col, when, max as spark_max, abs as spark_abs
)

df_silver = spark.table(
    "workspace.greenhouse_silver.production_clean"
)

df_mass = df_silver.filter(
    (col("production_and_value") == "Production") &
    (col("uom").isin("Kilograms", "Pounds"))
)

comparison = (
    df_mass
    .groupBy("ref_date", "geo", "commodity")
    .agg(
        spark_max(
            when(col("uom") == "Kilograms", col("value"))
        ).alias("reported_kg"),
        spark_max(
            when(col("uom") == "Pounds", col("value"))
        ).alias("reported_lb")
    )
    .filter(
        col("reported_kg").isNotNull() &
        col("reported_lb").isNotNull()
    )
    .withColumn(
        "converted_lb_to_kg",
        col("reported_lb") * 0.45359237
    )
    .withColumn(
        "difference_kg",
        spark_abs(
            col("reported_kg") - col("converted_lb_to_kg")
        )
    )
)

display(
    comparison.select(
        "ref_date",
        "geo",
        "commodity",
        "reported_kg",
        "reported_lb",
        "converted_lb_to_kg",
        "difference_kg"
    ).orderBy("difference_kg").limit(10)
)

print("Comparable pairs:", comparison.count())


ref_date,geo,commodity,reported_kg,reported_lb,converted_lb_to_kg,difference_kg
2016,Saskatchewan,Fresh Chinese vegetables,0.0,0.0,0.0,0.0
2016,Nova Scotia,Fresh green and wax beans [114221621],0.0,0.0,0.0,0.0
2016,New Brunswick,Fresh Chinese vegetables,0.0,0.0,0.0,0.0
2016,Alberta,Fresh Chinese vegetables,0.0,0.0,0.0,0.0
2016,Prince Edward Island,Fresh Chinese vegetables,0.0,0.0,0.0,0.0
2016,Prince Edward Island,Fresh fine herbs [11422138],0.0,0.0,0.0,0.0
2008,Newfoundland and Labrador,Fresh peppers [114221541],0.0,0.0,0.0,0.0
2016,Prince Edward Island,Fresh green and wax beans [114221621],0.0,0.0,0.0,0.0
2016,Manitoba,Fresh green and wax beans [114221621],0.0,0.0,0.0,0.0
2016,British Columbia,Fresh fine herbs [11422138],0.0,0.0,0.0,0.0


Comparable pairs: 1020


In [0]:
from pyspark.sql.functions import coalesce

df_gold = comparison.withColumn(
    "production_kg",
    coalesce(
        col("reported_kg"),
        col("reported_lb") * 0.45359237
    )
)

In [0]:

display(
    comparison
    .filter(
        (col("reported_kg") > 0) &
        (col("reported_lb") > 0)
    )
    .select(
        "ref_date",
        "geo",
        "commodity",
        "reported_kg",
        "reported_lb",
        "converted_lb_to_kg",
        "difference_kg"
    )
    .orderBy(col("difference_kg").desc())
    .limit(10)
)


ref_date,geo,commodity,reported_kg,reported_lb,converted_lb_to_kg,difference_kg
2025,Canada,Fresh cucumbers [11422152],3.8408506E8,8.46762556E8,3.840850346032977E8,25.3967022895813
2024,Canada,Fresh cucumbers [11422152],3.65219371E8,8.05170833E8,3.6521934639534426E8,24.604655742645264
2025,Canada,Fresh tomatoes [11422151],3.34677821E8,7.37838246E8,3.3467779867978305E8,22.320216953754425
2023,Canada,Fresh cucumbers [11422152],3.18254322E8,7.0163063E8,3.182543003262931E8,21.673706889152527
2024,Canada,Fresh tomatoes [11422151],3.20326951E8,7.06199996E8,3.2032692987963057E8,21.12036943435669
2025,Ontario,Fresh cucumbers [11422152],3.01336447E8,6.64333103E8,3.0133642665922415E8,20.340775847434998
2023,Canada,Fresh tomatoes [11422151],3.05264526E8,6.72993035E8,3.0526450573914295E8,20.260857045650482
2022,Canada,Fresh tomatoes [11422151],2.94408354E8,6.49059273E8,2.94408333910547E8,20.089452981948853
2018,Canada,Fresh tomatoes [11422151],2.79432751E8,6.16043723E8,2.794327323391935E8,18.660806477069855
2024,Ontario,Fresh cucumbers [11422152],2.80802467E8,6.1906343E8,2.8080244839402914E8,18.605970859527588


In [0]:

from pyspark.sql.functions import col, max as spark_max

positive_pairs = comparison.filter(
    (col("reported_kg") > 0) &
    (col("reported_lb") > 0)
)

validation = positive_pairs.withColumn(
    "relative_difference_pct",
    (col("difference_kg") / col("reported_kg")) * 100
)

print("Positive comparable pairs:", validation.count())

print(
    "Pairs with difference above 0.1%:",
    validation.filter(
        col("relative_difference_pct") > 0.1
    ).count()
)

display(
    validation.agg(
        spark_max("relative_difference_pct").alias(
            "maximum_relative_difference_pct"
        )
    )
)


Positive comparable pairs: 860
Pairs with difference above 0.1%: 14


maximum_relative_difference_pct
2.0582832500000148


In [0]:

display(
    validation
    .filter(col("relative_difference_pct") > 0.1)
    .select(
        "ref_date",
        "geo",
        "commodity",
        "reported_kg",
        "reported_lb",
        "difference_kg",
        "relative_difference_pct"
    )
    .orderBy(col("relative_difference_pct").desc())
)


ref_date,geo,commodity,reported_kg,reported_lb,difference_kg,relative_difference_pct
2017,Prince Edward Island,Fresh microgreens and shoots,16.0,36.0,0.32932532000000236,2.0582832500000148
2022,Nova Scotia,Fresh green and wax beans [114221621],25.0,56.0,0.4011727200000017,1.6046908800000068
2023,Nova Scotia,Fresh green and wax beans [114221621],26.0,58.0,0.30835745999999986,1.1859902307692303
2019,Nova Scotia,Fresh green and wax beans [114221621],70.0,155.0,0.3068173500000029,0.4383105000000041
2023,Prince Edward Island,Fresh strawberries [1141111],113.0,250.0,0.3980925000000042,0.3522942477876143
2018,Nova Scotia,Fresh green and wax beans [114221621],52.0,115.0,0.1631225500000042,0.31369721153846963
2017,Newfoundland and Labrador,Fresh Chinese vegetables,186.0,411.0,0.42646407000000863,0.22928175806452075
2024,New Brunswick,Fresh strawberries [1141111],80.0,176.0,0.1677428799999916,0.20967859999998947
2007,Prince Edward Island,Fresh peppers [114221541],200.0,440.0,0.419357199999979,0.20967859999998947
2020,Alberta,Fresh strawberries [1141111],318.0,700.0,0.48534100000000535,0.15262295597484446


In [0]:

from pyspark.sql.functions import max as spark_max

exceptions = validation.filter(
    col("relative_difference_pct") > 0.1
)

display(
    exceptions.agg(
        spark_max("difference_kg").alias(
            "max_absolute_difference_kg"
        )
    )
)

print("Exception records:", exceptions.count())
print(
    "Exceptions above 0.5 kg:",
    exceptions.filter(col("difference_kg") > 0.5).count()
)


max_absolute_difference_kg
0.48534100000000535


Exception records: 14
Exceptions above 0.5 kg: 0


In [0]:

from pyspark.sql import functions as F

df_silver = spark.table(
    "workspace.greenhouse_silver.production_clean"
)

# Only mass-based production measurements
df_mass = df_silver.filter(
    (F.col("production_and_value") == "Production") &
    (F.col("uom").isin("Kilograms", "Pounds"))
)

# Check that each unit occurs only once per business key
duplicates = (
    df_mass
    .groupBy("ref_date", "geo", "commodity", "uom")
    .count()
    .filter(F.col("count") > 1)
)

duplicate_count = duplicates.count()
print("Duplicate measurement keys:", duplicate_count)

assert duplicate_count == 0, "Review duplicate measurement keys"

# Bring both units into one row
df_pivoted = (
    df_mass
    .groupBy("ref_date", "geo", "commodity")
    .agg(
        F.max(
            F.when(F.col("uom") == "Kilograms", F.col("value"))
        ).alias("reported_kg"),
        F.max(
            F.when(F.col("uom") == "Pounds", F.col("value"))
        ).alias("reported_lb")
    )
)

# Select one standardized mass measurement
df_gold_mass = (
    df_pivoted
    .withColumn(
        "production_kg",
        F.coalesce(
            F.col("reported_kg"),
            F.col("reported_lb") * F.lit(0.45359237)
        )
    )
    .withColumn(
        "source_unit",
        F.when(F.col("reported_kg").isNotNull(), "Kilograms")
         .when(F.col("reported_lb").isNotNull(), "Pounds")
         .otherwise("MISSING")
    )
)

display(
    df_gold_mass
    .filter(F.col("production_kg") > 0)
    .orderBy("ref_date", "geo", "commodity")
    .limit(10)
)

print("Gold-ready mass records:", df_gold_mass.count())


Duplicate measurement keys: 0


ref_date,geo,commodity,reported_kg,reported_lb,production_kg,source_unit
1955,Alberta,Fresh tomatoes [11422151],null,177671.0,80590.20997027,Pounds
1955,British Columbia,Fresh tomatoes [11422151],null,1741049.0,789726.54219613,Pounds
1955,Canada,Fresh tomatoes [11422151],null,3547068.0,1608922.9806711602,Pounds
1955,Newfoundland and Labrador and Prince Edward Island,Fresh tomatoes [11422151],null,9300.0,4218.409041,Pounds
1955,Nova Scotia,Fresh tomatoes [11422151],null,57755.0,26197.22732935,Pounds
1955,Ontario,Fresh tomatoes [11422151],null,1513630.0,686571.0190031,Pounds
1955,Quebec,Fresh tomatoes [11422151],null,21645.0,9818.00684865,Pounds
1955,Saskatchewan,Fresh tomatoes [11422151],null,20518.0,9306.808247660001,Pounds
1956,Alberta,Fresh tomatoes [11422151],null,174593.0,79194.05265541,Pounds
1956,British Columbia,Fresh tomatoes [11422151],null,1407835.0,638583.21421895,Pounds


Gold-ready mass records: 2299


In [0]:

from pyspark.sql import functions as F

display(
    df_gold_mass
    .groupBy("source_unit")
    .agg(
        F.count("*").alias("total_records"),
        F.count("production_kg").alias("valid_quantities"),
        F.sum(
            F.when(
                F.col("production_kg").isNull(), 1
            ).otherwise(0)
        ).alias("missing_quantities")
    )
    .orderBy("source_unit")
)


source_unit,total_records,valid_quantities,missing_quantities
Kilograms,1020,1020,0
MISSING,823,0,823
Pounds,456,456,0


In [0]:

from pyspark.sql import functions as F

df_silver = spark.table(
    "workspace.greenhouse_silver.production_clean"
)

# Get quality classifications for each unit
df_unit_quality = (
    df_silver
    .filter(
        (F.col("production_and_value") == "Production") &
        (F.col("uom").isin("Kilograms", "Pounds"))
    )
    .groupBy("ref_date", "geo", "commodity")
    .agg(
        F.max(
            F.when(
                F.col("uom") == "Kilograms",
                F.col("measurement_quality")
            )
        ).alias("kg_quality"),

        F.max(
            F.when(
                F.col("uom") == "Pounds",
                F.col("measurement_quality")
            )
        ).alias("lb_quality")
    )
)

# Attach quality information to missing Gold records
df_missing = (
    df_gold_mass
    .filter(F.col("production_kg").isNull())
    .join(
        df_unit_quality,
        ["ref_date", "geo", "commodity"],
        "left"
    )
)

display(
    df_missing
    .groupBy("kg_quality", "lb_quality")
    .count()
    .orderBy(F.desc("count"))
)

print("Missing production groups:", df_missing.count())


kg_quality,lb_quality,count
CONFIDENTIAL,CONFIDENTIAL,364
NOT_AVAILABLE,NOT_AVAILABLE,312
null,CONFIDENTIAL,76
UNRELIABLE,UNRELIABLE,68
NOT_AVAILABLE,CONFIDENTIAL,3


Missing production groups: 823


In [0]:

from pyspark.sql import functions as F

df_gold_quality = (
    df_gold_mass
    .join(
        df_unit_quality,
        ["ref_date", "geo", "commodity"],
        "left"
    )
    .withColumn(
        "production_quality",
        F.when(
            F.col("production_kg").isNotNull(),
            F.when(
                F.col("source_unit") == "Kilograms",
                F.col("kg_quality")
            ).otherwise(F.col("lb_quality"))
        )
        .when(
            (F.col("kg_quality") == "CONFIDENTIAL") |
            (F.col("lb_quality") == "CONFIDENTIAL"),
            "CONFIDENTIAL"
        )
        .when(
            (F.col("kg_quality") == "NOT_AVAILABLE") |
            (F.col("lb_quality") == "NOT_AVAILABLE"),
            "NOT_AVAILABLE"
        )
        .when(
            (F.col("kg_quality") == "UNRELIABLE") |
            (F.col("lb_quality") == "UNRELIABLE"),
            "UNRELIABLE"
        )
        .otherwise("REVIEW_REQUIRED")
    )
)

display(
    df_gold_quality
    .groupBy("production_quality")
    .count()
    .orderBy("production_quality")
)

print("Total Gold groups:", df_gold_quality.count())


production_quality,count
AVAILABLE,1347
CONFIDENTIAL,443
NOT_AVAILABLE,312
UNRELIABLE,68
USE_WITH_CAUTION,129


Total Gold groups: 2299


In [0]:

from pyspark.sql import functions as F

# 1. Check duplicate business keys
duplicate_count = (
    df_gold_quality
    .groupBy("ref_date", "geo", "commodity")
    .count()
    .filter(F.col("count") > 1)
    .count()
)

# 2. Check quantity and quality consistency
quality_mismatches = df_gold_quality.filter(
    (
        F.col("production_kg").isNotNull() &
        ~F.col("production_quality").isin(
            "AVAILABLE", "USE_WITH_CAUTION"
        )
    ) |
    (
        F.col("production_kg").isNull() &
        ~F.col("production_quality").isin(
            "CONFIDENTIAL",
            "NOT_AVAILABLE",
            "UNRELIABLE"
        )
    ) |
    F.col("production_quality").isNull()
).count()

# 3. Verify the record count
total_groups = df_gold_quality.count()

print("Duplicate business keys:", duplicate_count)
print("Quality mismatches:", quality_mismatches)
print("Total Gold groups:", total_groups)

# Stop processing if any check fails
assert duplicate_count == 0
assert quality_mismatches == 0
assert total_groups == 2299

print("Gold validation: PASSED")


Duplicate business keys: 0
Quality mismatches: 0
Total Gold groups: 2299
Gold validation: PASSED


In [0]:

from pyspark.sql import functions as F

# 1. Prepare the final Gold fact table
df_gold_fact = (
    df_gold_quality
    .select(
        F.col("ref_date").alias("year"),
        F.col("geo").alias("geography"),
        "commodity",
        "production_kg",
        "production_quality",
        "source_unit",
        "reported_kg",
        "reported_lb",
        "kg_quality",
        "lb_quality"
    )
    .withColumn(
        "is_canada_total",
        F.col("geography") == "Canada"
    )
)

# 2. Save as a permanent Delta table
(
    df_gold_fact.write
    .format("delta")
    .mode("overwrite")
    .option("overwriteSchema", "true")
    .saveAsTable(
        "workspace.greenhouse_gold.fact_production_mass"
    )
)

# 3. Verify the saved table
df_saved_gold = spark.table(
    "workspace.greenhouse_gold.fact_production_mass"
)

print("Saved Gold records:", df_saved_gold.count())
print("Gold fact table created successfully")


Saved Gold records: 2299
Gold fact table created successfully


In [0]:

%sql
SELECT
    year,
    geography,
    commodity,
    ROUND(production_kg, 2) AS production_kg,
    production_quality
FROM workspace.greenhouse_gold.fact_production_mass
WHERE geography = 'Canada'
  AND commodity LIKE 'Fresh tomatoes%'
  AND year IN (2024, 2025)
ORDER BY year;


year,geography,commodity,production_kg,production_quality
2024,Canada,Fresh tomatoes [11422151],3.20326951E8,AVAILABLE
2025,Canada,Fresh tomatoes [11422151],3.34677821E8,AVAILABLE


In [0]:


%sql
WITH tomato_production AS (
    SELECT
        year,
        production_kg,
        production_quality,
        LAG(production_kg) OVER (
            ORDER BY year
        ) AS previous_year_kg
    FROM workspace.greenhouse_gold.fact_production_mass
    WHERE geography = 'Canada'
      AND commodity LIKE 'Fresh tomatoes%'
      AND year IN (2024, 2025)
)

SELECT
    year,
    production_kg,
    previous_year_kg,
    production_quality,
    ROUND(
        (production_kg - previous_year_kg)
        / previous_year_kg * 100,
        2
    ) AS yoy_growth_pct
FROM tomato_production
ORDER BY year;


year,production_kg,previous_year_kg,production_quality,yoy_growth_pct
2024,3.20326951E8,null,AVAILABLE,null
2025,3.34677821E8,3.20326951E8,AVAILABLE,4.48


In [0]:
%sql



WITH production_history AS (
    SELECT
        year,
        geography,
        commodity,
        production_kg,
        production_quality,

        LAG(year) OVER (
            PARTITION BY geography, commodity
            ORDER BY year
        ) AS previous_year,

        LAG(production_kg) OVER (
            PARTITION BY geography, commodity
            ORDER BY year
        ) AS previous_year_kg,

        LAG(production_quality) OVER (
            PARTITION BY geography, commodity
            ORDER BY year
        ) AS previous_quality

    FROM workspace.greenhouse_gold.fact_production_mass
    WHERE geography IN ('Ontario', 'Quebec')
      AND commodity LIKE 'Fresh tomatoes%'
)

SELECT
    year,
    geography,
    commodity,
    production_quality,
    previous_quality,

    CASE
        WHEN previous_year = year - 1
         AND previous_year_kg > 0
         AND production_quality = 'AVAILABLE'
         AND previous_quality = 'AVAILABLE'
        THEN ROUND(
            (production_kg - previous_year_kg)
            / previous_year_kg * 100,
            2
        )
        ELSE NULL
    END AS approved_yoy_growth_pct

FROM production_history
WHERE year = 2025
ORDER BY geography;



year,geography,commodity,production_quality,previous_quality,approved_yoy_growth_pct
2025,Ontario,Fresh tomatoes [11422151],AVAILABLE,AVAILABLE,1.22
2025,Quebec,Fresh tomatoes [11422151],USE_WITH_CAUTION,AVAILABLE,null


In [0]:
%sql

CREATE OR REPLACE VIEW
workspace.greenhouse_gold.vw_production_yoy
AS

WITH production_history AS (
    SELECT
        year,
        geography,
        commodity,
        is_canada_total,
        production_kg,
        production_quality,

        LAG(year) OVER (
            PARTITION BY geography, commodity
            ORDER BY year
        ) AS previous_year,

        LAG(production_kg) OVER (
            PARTITION BY geography, commodity
            ORDER BY year
        ) AS previous_year_kg,

        LAG(production_quality) OVER (
            PARTITION BY geography, commodity
            ORDER BY year
        ) AS previous_quality

    FROM workspace.greenhouse_gold.fact_production_mass
)

SELECT
    year,
    geography,
    commodity,
    is_canada_total,
    production_kg,
    production_quality,
    previous_year,
    previous_year_kg,
    previous_quality,

    CASE
        WHEN previous_year = year - 1
         AND previous_year_kg > 0
         AND production_quality = 'AVAILABLE'
         AND previous_quality = 'AVAILABLE'
        THEN ROUND(
            (production_kg - previous_year_kg)
            / previous_year_kg * 100,
            2
        )
        ELSE NULL
    END AS approved_yoy_growth_pct

FROM production_history;


In [0]:
%sql
CREATE OR REPLACE VIEW
workspace.greenhouse_gold.vw_production_yoy
AS

WITH production_history AS (
    SELECT
        year,
        geography,
        commodity,
        is_canada_total,
        production_kg,
        production_quality,

        LAG(year) OVER (
            PARTITION BY geography, commodity
            ORDER BY year
        ) AS previous_year,

        LAG(production_kg) OVER (
            PARTITION BY geography, commodity
            ORDER BY year
        ) AS previous_year_kg,

        LAG(production_quality) OVER (
            PARTITION BY geography, commodity
            ORDER BY year
        ) AS previous_quality

    FROM workspace.greenhouse_gold.fact_production_mass
)

SELECT
    year,
    geography,
    commodity,
    is_canada_total,
    production_kg,
    production_quality,
    previous_year,
    previous_year_kg,
    previous_quality,

    CASE
        WHEN previous_year = year - 1
         AND previous_year_kg > 0
         AND production_quality = 'AVAILABLE'
         AND previous_quality = 'AVAILABLE'
        THEN ROUND(
            (production_kg - previous_year_kg)
            / previous_year_kg * 100,
            2
        )
        ELSE NULL
    END AS approved_yoy_growth_pct

FROM production_history;

In [0]:
%sql
CREATE OR REPLACE VIEW
workspace.greenhouse_gold.vw_production_yoy
AS

WITH production_history AS (
    SELECT
        year,
        geography,
        commodity,
        is_canada_total,
        production_kg,
        production_quality,

        LAG(year) OVER (
            PARTITION BY geography, commodity
            ORDER BY year
        ) AS previous_year,

        LAG(production_kg) OVER (
            PARTITION BY geography, commodity
            ORDER BY year
        ) AS previous_year_kg,

        LAG(production_quality) OVER (
            PARTITION BY geography, commodity
            ORDER BY year
        ) AS previous_quality

    FROM workspace.greenhouse_gold.fact_production_mass
)

SELECT
    year,
    geography,
    commodity,
    is_canada_total,
    production_kg,
    production_quality,
    previous_year,
    previous_year_kg,
    previous_quality,

    CASE
        WHEN previous_year = year - 1
         AND previous_year_kg > 0
         AND production_quality = 'AVAILABLE'
         AND previous_quality = 'AVAILABLE'
        THEN ROUND(
            (production_kg - previous_year_kg)
            / previous_year_kg * 100,
            2
        )
        ELSE NULL
    END AS approved_yoy_growth_pct

FROM production_history;

In [0]:
%sql
SELECT
    year,
    geography,
    commodity,
    production_quality,
    approved_yoy_growth_pct
FROM workspace.greenhouse_gold.vw_production_yoy
WHERE year = 2025
  AND geography IN ('Ontario', 'Quebec')
  AND commodity LIKE 'Fresh tomatoes%'
ORDER BY geography;

year,geography,commodity,production_quality,approved_yoy_growth_pct
2025,Ontario,Fresh tomatoes [11422151],AVAILABLE,1.22
2025,Quebec,Fresh tomatoes [11422151],USE_WITH_CAUTION,null


In [0]:
%sql
SELECT
    ref_date,
    geo,
    commodity,
    COUNT(*) AS record_count
FROM workspace.greenhouse_silver.production_clean
WHERE production_and_value = 'Farm gate value'
GROUP BY
    ref_date,
    geo,
    commodity
HAVING COUNT(*) > 1
ORDER BY record_count DESC;

ref_date,geo,commodity,record_count


In [0]:
%sql
CREATE OR REPLACE TABLE
workspace.greenhouse_gold.fact_farm_gate_value
AS

SELECT
    ref_date AS year,
    geo AS geography,
    commodity,
    value AS farm_gate_value_cad,
    measurement_quality AS value_quality,
    status AS source_status,
    CASE
        WHEN geo = 'Canada' THEN TRUE
        ELSE FALSE
    END AS is_canada_total

FROM workspace.greenhouse_silver.production_clean

WHERE production_and_value = 'Farm gate value';

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
    COUNT(*) AS total_records,
    COUNT(farm_gate_value_cad) AS numeric_values,
    COUNT(*) - COUNT(farm_gate_value_cad) AS missing_values
FROM workspace.greenhouse_gold.fact_farm_gate_value;

total_records,numeric_values,missing_values
3048,1876,1172


In [0]:
%sql
WITH area_pairs AS (
    SELECT
        ref_date,
        geo,
        commodity,

        MAX(
            CASE
                WHEN uom = 'Square metres'
                THEN value
            END
        ) AS reported_m2,

        MAX(
            CASE
                WHEN uom = 'Square feet'
                THEN value
            END
        ) AS reported_sqft

    FROM workspace.greenhouse_silver.production_clean

    WHERE production_and_value = 'Area harvested'

    GROUP BY
        ref_date,
        geo,
        commodity
)

SELECT
    COUNT(*) AS total_area_groups,

    SUM(
        CASE
            WHEN reported_m2 IS NOT NULL
             AND reported_sqft IS NOT NULL
            THEN 1
            ELSE 0
        END
    ) AS groups_with_both_units

FROM area_pairs;

total_area_groups,groups_with_both_units
2050,1183


In [0]:
%sql
WITH area_pairs AS (
    SELECT
        ref_date,
        geo,
        commodity,

        MAX(
            CASE WHEN uom = 'Square metres'
            THEN value END
        ) AS reported_m2,

        MAX(
            CASE WHEN uom = 'Square feet'
            THEN value END
        ) AS reported_sqft

    FROM workspace.greenhouse_silver.production_clean

    WHERE production_and_value = 'Area harvested'

    GROUP BY
        ref_date,
        geo,
        commodity
),

comparison AS (
    SELECT
        ref_date,
        geo,
        commodity,
        reported_m2,
        reported_sqft,
        reported_sqft * 0.09290304 AS converted_sqft_to_m2,

        ABS(
            reported_m2 - (reported_sqft * 0.09290304)
        ) AS difference_m2,

        CASE
            WHEN reported_m2 > 0
            THEN
                ABS(
                    reported_m2 - (reported_sqft * 0.09290304)
                ) / reported_m2 * 100
        END AS relative_difference_pct

    FROM area_pairs

    WHERE reported_m2 IS NOT NULL
      AND reported_sqft IS NOT NULL
)

SELECT
    COUNT(*) AS comparable_pairs,

    SUM(
        CASE
            WHEN relative_difference_pct > 0.1
            THEN 1
            ELSE 0
        END
    ) AS pairs_above_0_1_pct,

    MAX(relative_difference_pct)
        AS max_relative_difference_pct

FROM comparison;

comparable_pairs,pairs_above_0_1_pct,max_relative_difference_pct
1183,49,2.4518079999999998


In [0]:
%sql
WITH area_pairs AS (
    SELECT
        ref_date,
        geo,
        commodity,
        MAX(CASE WHEN uom = 'Square metres' THEN value END) AS reported_m2,
        MAX(CASE WHEN uom = 'Square feet' THEN value END) AS reported_sqft
    FROM workspace.greenhouse_silver.production_clean
    WHERE production_and_value = 'Area harvested'
    GROUP BY ref_date, geo, commodity
),

comparison AS (
    SELECT
        ref_date,
        geo,
        commodity,
        reported_m2,
        reported_sqft,
        reported_sqft * 0.09290304 AS converted_sqft_to_m2,
        ABS(
            reported_m2 - (reported_sqft * 0.09290304)
        ) AS difference_m2
    FROM area_pairs
    WHERE reported_m2 IS NOT NULL
      AND reported_sqft IS NOT NULL
)

SELECT
    MAX(difference_m2) AS max_absolute_difference_m2,
    SUM(
        CASE WHEN difference_m2 > 1 THEN 1 ELSE 0 END
    ) AS pairs_above_1_m2
FROM comparison;

max_absolute_difference_m2,pairs_above_1_m2
2.6050304006785154,17


In [0]:
%sql
CREATE OR REPLACE TABLE
workspace.greenhouse_gold.fact_harvested_area
AS

WITH area_source AS (
    SELECT
        ref_date,
        geo,
        commodity,

        MAX(
            CASE WHEN uom = 'Square metres'
            THEN value END
        ) AS reported_m2,

        MAX(
            CASE WHEN uom = 'Square feet'
            THEN value END
        ) AS reported_sqft,

        MAX(
            CASE WHEN uom = 'Square metres'
            THEN measurement_quality END
        ) AS m2_quality,

        MAX(
            CASE WHEN uom = 'Square feet'
            THEN measurement_quality END
        ) AS sqft_quality

    FROM workspace.greenhouse_silver.production_clean
    WHERE production_and_value = 'Area harvested'

    GROUP BY
        ref_date,
        geo,
        commodity
)

SELECT
    ref_date AS year,
    geo AS geography,
    commodity,

    reported_m2,
    reported_sqft,

    COALESCE(
        reported_m2,
        reported_sqft * 0.09290304
    ) AS harvested_area_m2,

    CASE
        WHEN reported_m2 IS NOT NULL
            THEN 'Square metres'
        WHEN reported_sqft IS NOT NULL
            THEN 'Square feet'
        ELSE 'MISSING'
    END AS source_unit,

    m2_quality,
    sqft_quality,

    CASE
        WHEN reported_m2 IS NOT NULL
            THEN m2_quality

        WHEN reported_sqft IS NOT NULL
            THEN sqft_quality

        WHEN m2_quality = 'CONFIDENTIAL'
          OR sqft_quality = 'CONFIDENTIAL'
            THEN 'CONFIDENTIAL'

        WHEN m2_quality = 'NOT_AVAILABLE'
          OR sqft_quality = 'NOT_AVAILABLE'
            THEN 'NOT_AVAILABLE'

        WHEN m2_quality = 'UNRELIABLE'
          OR sqft_quality = 'UNRELIABLE'
            THEN 'UNRELIABLE'

        ELSE 'REVIEW_REQUIRED'
    END AS area_quality,

    CASE
        WHEN geo = 'Canada' THEN TRUE
        ELSE FALSE
    END AS is_canada_total

FROM area_source;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
    COUNT(*) AS total_records,
    COUNT(harvested_area_m2) AS numeric_values,
    COUNT(*) - COUNT(harvested_area_m2) AS missing_values,
    SUM(
        CASE WHEN area_quality = 'REVIEW_REQUIRED'
        THEN 1 ELSE 0 END
    ) AS review_required
FROM workspace.greenhouse_gold.fact_harvested_area;

total_records,numeric_values,missing_values,review_required
2050,1183,867,0


In [0]:
%sql
WITH price_pairs AS (
    SELECT
        ref_date,
        geo,
        commodity,

        MAX(
            CASE WHEN uom = 'Dollars per kilogram'
            THEN value END
        ) AS reported_per_kg,

        MAX(
            CASE WHEN uom = 'Dollars per pound'
            THEN value END
        ) AS reported_per_lb

    FROM workspace.greenhouse_silver.production_clean
    WHERE production_and_value = 'Average price'

    GROUP BY
        ref_date,
        geo,
        commodity
),

comparison AS (
    SELECT
        ref_date,
        geo,
        commodity,
        reported_per_kg,
        reported_per_lb,

        reported_per_lb * 2.20462262
            AS converted_lb_price_to_kg,

        CASE
            WHEN reported_per_kg > 0
            THEN ABS(
                reported_per_kg -
                (reported_per_lb * 2.20462262)
            ) / reported_per_kg * 100
        END AS relative_difference_pct

    FROM price_pairs
    WHERE reported_per_kg IS NOT NULL
      AND reported_per_lb IS NOT NULL
)

SELECT
    COUNT(*) AS comparable_pairs,

    SUM(
        CASE
            WHEN relative_difference_pct > 1
            THEN 1
            ELSE 0
        END
    ) AS pairs_above_1_pct,

    MAX(relative_difference_pct)
        AS max_relative_difference_pct

FROM comparison;

comparable_pairs,pairs_above_1_pct,max_relative_difference_pct
880,0,0.7827483428571393


In [0]:
%sql
CREATE OR REPLACE TABLE
workspace.greenhouse_gold.fact_average_price_mass
AS

WITH price_source AS (
    SELECT
        ref_date,
        geo,
        commodity,

        MAX(
            CASE WHEN uom = 'Dollars per kilogram'
            THEN value END
        ) AS reported_price_per_kg,

        MAX(
            CASE WHEN uom = 'Dollars per pound'
            THEN value END
        ) AS reported_price_per_lb,

        MAX(
            CASE WHEN uom = 'Dollars per kilogram'
            THEN measurement_quality END
        ) AS kg_price_quality,

        MAX(
            CASE WHEN uom = 'Dollars per pound'
            THEN measurement_quality END
        ) AS lb_price_quality

    FROM workspace.greenhouse_silver.production_clean

    WHERE production_and_value = 'Average price'
      AND uom IN (
          'Dollars per kilogram',
          'Dollars per pound'
      )

    GROUP BY
        ref_date,
        geo,
        commodity
)

SELECT
    ref_date AS year,
    geo AS geography,
    commodity,

    reported_price_per_kg,
    reported_price_per_lb,

    COALESCE(
        reported_price_per_kg,
        reported_price_per_lb * 2.20462262
    ) AS average_price_cad_per_kg,

    CASE
        WHEN reported_price_per_kg IS NOT NULL
            THEN 'Dollars per kilogram'
        WHEN reported_price_per_lb IS NOT NULL
            THEN 'Dollars per pound'
        ELSE 'MISSING'
    END AS source_unit,

    kg_price_quality,
    lb_price_quality,

    CASE
        WHEN reported_price_per_kg IS NOT NULL
            THEN kg_price_quality

        WHEN reported_price_per_lb IS NOT NULL
            THEN lb_price_quality

        WHEN kg_price_quality = 'CONFIDENTIAL'
          OR lb_price_quality = 'CONFIDENTIAL'
            THEN 'CONFIDENTIAL'

        WHEN kg_price_quality = 'NOT_AVAILABLE'
          OR lb_price_quality = 'NOT_AVAILABLE'
            THEN 'NOT_AVAILABLE'

        WHEN kg_price_quality = 'UNRELIABLE'
          OR lb_price_quality = 'UNRELIABLE'
            THEN 'UNRELIABLE'

        ELSE 'REVIEW_REQUIRED'
    END AS price_quality,

    CASE
        WHEN geo = 'Canada' THEN TRUE
        ELSE FALSE
    END AS is_canada_total

FROM price_source;

num_affected_rows,num_inserted_rows


In [0]:
%sql
SELECT
    COUNT(*) AS total_records,
    COUNT(average_price_cad_per_kg) AS numeric_values,
    COUNT(*) - COUNT(average_price_cad_per_kg) AS missing_values,
    SUM(
        CASE WHEN price_quality = 'REVIEW_REQUIRED'
        THEN 1 ELSE 0 END
    ) AS review_required
FROM workspace.greenhouse_gold.fact_average_price_mass;

total_records,numeric_values,missing_values,review_required
2299,1341,958,0


In [0]:
%sql
CREATE OR REPLACE VIEW
workspace.greenhouse_gold.vw_greenhouse_analytics
AS

SELECT
    p.year,
    p.geography,
    p.commodity,
    p.is_canada_total,

    p.production_kg,
    p.production_quality,

    f.farm_gate_value_cad,
    f.value_quality AS farm_gate_quality,

    a.harvested_area_m2,
    a.area_quality,

    pr.average_price_cad_per_kg,
    pr.price_quality

FROM workspace.greenhouse_gold.fact_production_mass p

LEFT JOIN workspace.greenhouse_gold.fact_farm_gate_value f
    ON p.year = f.year
   AND p.geography = f.geography
   AND p.commodity = f.commodity

LEFT JOIN workspace.greenhouse_gold.fact_harvested_area a
    ON p.year = a.year
   AND p.geography = a.geography
   AND p.commodity = a.commodity

LEFT JOIN workspace.greenhouse_gold.fact_average_price_mass pr
    ON p.year = pr.year
   AND p.geography = pr.geography
   AND p.commodity = pr.commodity;

In [0]:
%sql
SELECT
    COUNT(*) AS total_rows
FROM workspace.greenhouse_gold.vw_greenhouse_analytics;

total_rows
2299
